# MASLD-LLM · 03 — Fine-tuning (LoRA/QLoRA) on Colab Pro

The compute-heavy notebook. Paper recipe (STAR Methods, p.e4–e5) translated to single-GPU Colab:

| Paper (8×GPU) | Colab translation |
|---|---|
| Qwen3.5-9B + LoRA r32 α32, 2 stages, 28 h wall | Qwen3.5-9B **QLoRA r32 α32**, seq packing, grad ckpt, 2 stages, ~2–4 h/run on A100 40GB |
| Full-FT of 0.6B/1.7B on 8×H100 | LoRA on 0.6B/1.7B on **L4** — often under an hour |
| LLaMA-Factory + DeepSpeed ZeRO-2 | **LLaMA-Factory** (single GPU, no DeepSpeed) or unsloth |
| 32k context, packing, assistant-only loss | identical — this is what LLaMA-Factory does for you |

Prerequisites: notebook 01 has written the frozen prompt JSONLs to `Drive/masld/prompts/`.

## Compute-unit budget (A100 ≈ 8–12 CU/hr on Pro+)

| Run | Est. CU | Sessions |
|---|---|---|
| 9B QLoRA stage A (1 ep, pretrain corpus) | ~25 CU | 1× 3 h |
| 9B QLoRA stage B (3 ep, SFT) | ~40 CU | 1–2× 4 h |
| vLLM eval of your model | ~6 CU | 1× 1 h |
| Baseline + frontier evals | ~0 (API) | any runtime |

→ The whole 9B program fits inside one Pro+ month (500 CU). The 0.6/1.7B ladder fits inside plain Pro (100 CU).
## v2 — data-volume budget (hierarchical-uniform sampling)

With the v2 corpora (Fujiwara + drugs + MSigDB + FerrDb + STRING + Tahoe), raw facts exceed 250k prompts / ~300M+ tokens. The paper's answer to source imbalance — and yours — is **hierarchical-uniform sampling**: balance first across domains, then across tasks within them, then cap total tokens seen.

| Domain (stage-B side) | Raw prompts | Sampled share |
|---|---|---|
| Liver clinical (Kamzolas + Fujiwara + panels) | ~60–80k | ~35% |
| Plasma + blood | ~3k | ~5% |
| Progression (paired biopsies) | ~120 | up to 3 epochs extra weight |
| Drug/pathway/PPI knowledge | ~100–150k | ~35% |
| Perturbation (Tahoe) | 60k | ~25% |

Cap: **~50–100M tokens seen** per SFT run (≈ 1 epoch on the sampled mixture, 3 epochs total). This keeps the 9B QLoRA inside one Pro+ month and actually helps the 0.6–1.7B rungs, where their own small-model gains came from balanced mixtures rather than raw scale.

Implementation: after concatenating the alpaca JSONs in §1, down-sample each domain to its share before writing `sft_multitask.json`; keep the unsampled pools for future runs.

## 0 · Runtime & install
Pro+ → Runtime → **A100 40GB** (if only V100/A100-80 unavailable, take L4 ×2 sessions and drop to the 1.7B ladder; 9B QLoRA really wants 40GB).
Session limits: Pro+ gives up to 24 h sessions; background execution keeps running with the tab closed — but **save checkpoints to Drive every epoch** (see `save_steps` below) because Colab can still reclaim the VM.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')%cd /content!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git%cd LLaMA-Factory!pip -q install -e ".[torch,metrics]"# QLoRA bits!pip -q install bitsandbytes peft transformers acceleratePROMPTS = '/content/drive/MyDrive/masld/prompts'CKPT    = '/content/drive/MyDrive/masld/checkpoints'import os; os.makedirs(CKPT, exist_ok=True)

## 1 · Convert frozen JSONLs → LLaMA-Factory alpaca-style JSON

In [ ]:
import json, gzip, osdef to_alpaca(in_path, out_path):    rows = []    with gzip.open(in_path,'rt') as f:        first = json.loads(f.readline())        lines = f if '__meta__' in first else None        if lines is None:            rows.append(first)        for l in (lines if lines else []):            rows.append(json.loads(l))    out = [dict(instruction=r['prompt'], output=r['answer']) for r in rows]    json.dump(out, open(out_path,'w'), indent=0)    print(out_path, len(out))to_alpaca(f'{PROMPTS}/MB-10xx_liver_train.jsonl.gz', '/content/sft_liver.json')to_alpaca(f'{PROMPTS}/MB-11xx_external_train.jsonl.gz', '/content/sft_external.json')to_alpaca(f'{PROMPTS}/MB-61xx_msigdb_train.jsonl.gz', '/content/sft_knowledge.json')to_alpaca(f'{PROMPTS}/MB-62xx_drugs_train.jsonl.gz', '/content/sft_drugs.json')to_alpaca(f'{PROMPTS}/MB-63xx_drugcentral_affinity_train.jsonl.gz', '/content/sft_drugs2.json')to_alpaca(f'{PROMPTS}/MB-70xx_dge_direction_train.jsonl.gz', '/content/sft_dge.json')to_alpaca(f'{PROMPTS}/MB-8xxx_fujiwara_train.jsonl.gz', '/content/sft_fujiwara.json')# repeat for the other train files; concatenate into one multitask SFT file:import functoolsfiles = ['/content/sft_liver.json', '/content/sft_external.json', '/content/sft_knowledge.json', '/content/sft_drugs.json', '/content/sft_dge.json', '/content/sft_fujiwara.json']  # all v3 train corporabig = []for fp in files: big += json.load(open(fp))json.dump(big, open('/content/sft_multitask.json','w'), indent=0)print('multitask SFT size:', len(big))

In [ ]:
# v2: hierarchical-uniform domain sampling before trainingimport json, randomrs = random.Random(20260929)DOMAIN_SHARE = {   # domain key -> (file, share)  'liver':   (['/content/sft_liver.json', '/content/sft_external.json', '/content/sft_fujiwara.json'], 0.35),  'knowledge': (['/content/sft_knowledge.json', '/content/sft_dge.json'], 0.15),  'drugs':   (['/content/sft_drugs.json', '/content/sft_drugs2.json'], 0.25),  'tahoe':   (['/content/sft_tahoe.json'], 0.25),   # placeholder until Tahoe corpus built; rebalance if absent}TARGET = 60_000   # prompts in the final mixture (paper-sized stage-B corpus)pool = {k: sum([json.load(open(f)) for f in fps], []) for k, (fps, _) in DOMAIN_SHARE.items()}out = []for k, (fp, share) in DOMAIN_SHARE.items():    n = int(TARGET * share)    rows = pool[k]    if len(rows) > n: rows = rs.sample(rows, n)    out += rowsrs.shuffle(out)json.dump(out, open('/content/sft_multitask.json', 'w'), indent=0)print('mixture size:', len(out), {k: len(v) for k, v in pool.items()})

## 2 · Dataset registration + LoRA config (Qwen3-0.6B on L4 first)
Shrink-first strategy: prove the pipeline end-to-end on 0.6B (minutes per epoch), then scale to 9B. Same hyperparameters the paper used, scaled for single GPU.

In [ ]:
# register datasetds_entry = '''{  "masld_sft": {    "file_name": "/content/sft_multitask.json",    "columns": {"prompt": "instruction", "response": "output"}  }}'''open('data/dataset_info.json','w').write(ds_entry)  # append, don't clobber, if you customize moreyaml_06b = '''### modelmodel_name_or_path: Qwen/Qwen3-0.6Btrust_remote_code: true### methodstage: sftdo_train: truefinetuning_type: loralora_rank: 32lora_alpha: 32lora_dropout: 0.0lisa_layers: 2### datasetdataset: masld_sfttemplate: qwencutoff_len: 8192            # paper used 32768; 8k covers MASLD profiles comfortablypacking: trueneat_packing: true          # isolates cross-sample attention (paper's 'neat packing')max_samples: 100000overwrite_cache: truepreprocessing_num_workers: 16### outputoutput_dir: /content/drive/MyDrive/masld/checkpoints/qwen3-0.6b-masld-sftlogging_steps: 10save_steps: 0.25            # save adapter every quarter epoch -> Driveplot_loss: true### trainper_device_train_batch_size: 2gradient_accumulation_steps: 16learning_rate: 1.0e-4       # paper: 1e-4 for SFT stage; 2e-5 for pretrain stagenum_train_epochs: 3.0lr_scheduler_type: cosinewarmup_ratio: 0.1weight_decay: 0.01bf16: truegradient_checkpointing: true'''open('/content/qwen06b_sft.yaml','w').write(yaml_06b)print('config written')

## 3 · Train (0.6B ladder rung)
Watch the first 20 steps: loss should drop from ~4–5 to <2 within 50 steps on packed data. If OOM → `cutoff_len: 4096` or `per_device_train_batch_size: 1`.

In [ ]:
!llamafactory-cli train /content/qwen06b_sft.yaml

## 4 · Scale to Qwen3.5-9B QLoRA on A100
Only after 0.6B works. Two stages, mirroring the paper: continued pretraining on the KG corpus, then multitask SFT initialized from the stage-A adapter.

> Qwen3.5-9B is hybrid GatedDeltaNet; ensure `transformers>=4.57` and flash-linear-attention kernels (`pip install fla-org` if needed). If the hybrid attention fights the Colab stack, fall back to **Qwen3-8B** — same family, standard attention, zero drama.

In [ ]:
yaml_9b_stageA = '''model_name_or_path: Qwen/Qwen3.5-9Bstage: sft                      # raw-text continuation via empty instruction works; or use 'pt'finetuning_type: loralora_rank: 32lora_alpha: 32lora_dropout: 0.0quantization_bit: 4             # QLoRAdataset: masld_pretrain         # kg_corpus.txt registered as text corpuscutoff_len: 16384packing: truetemplate: qwenper_device_train_batch_size: 1gradient_accumulation_steps: 32learning_rate: 2.0e-5           # paper stage-A LRnum_train_epochs: 1.0lr_scheduler_type: cosinewarmup_ratio: 0.1weight_decay: 0.01bf16: truegradient_checkpointing: trueoutput_dir: /content/drive/MyDrive/masld/checkpoints/qwen35-9b-masld-stageA'''open('/content/qwen9b_stageA.yaml','w').write(yaml_9b_stageA)print('stage A config written — run: llamafactory-cli train /content/qwen9b_stageA.yaml')

In [ ]:
yaml_9b_stageB = '''model_name_or_path: Qwen/Qwen3.5-9Badapter_name_or_path: /content/drive/MyDrive/masld/checkpoints/qwen35-9b-masld-stageAstage: sftfinetuning_type: loralora_rank: 32lora_alpha: 32lora_dropout: 0.0quantization_bit: 4dataset: masld_sftcutoff_len: 16384packing: trueneat_packing: truetemplate: qwenper_device_train_batch_size: 1gradient_accumulation_steps: 32learning_rate: 1.0e-4num_train_epochs: 3.0lr_scheduler_type: cosinewarmup_ratio: 0.1weight_decay: 0.01bf16: truegradient_checkpointing: trueoutput_dir: /content/drive/MyDrive/masld/checkpoints/qwen35-9b-masld-stageB'''open('/content/qwen9b_stageB.yaml','w').write(yaml_9b_stageB)print('stage B config — runs ~3-4 h on A100 40GB; save_steps to Drive')

## 5 · Merge & evaluate with vLLM (paper protocol)
Single-pass, temperature 0, top_p 1.0, max 1500 new tokens, served via vLLM.

In [ ]:
!pip -q install vllm# merge adapter into base weights (do on CPU or A100, not L4)!llamafactory-cli export merge \    --model_name_or_path Qwen/Qwen3.5-9B \    --adapter_name_or_path {CKPT}/qwen35-9b-masld-stageB \    --export_dir /content/drive/MyDrive/masld/model/qwen35-9b-masld-merged# then serve & score against the SAME frozen test JSONLs as notebook 02# (evaluate() from notebook 02, pointed at the local vLLM endpoint)

## 6 · Publish
- Dataset: `hf upload … masld-bench` (HF username → `masld-bench` dataset repo)
- Model: upload merged model or adapters → `masld-llm-qwen35-9b`
- Leaderboard: a static GitHub Page over `results/leaderboard.csv` — you already have `masld-clinical-transcriptomics-github` for the pipeline repo; add a `bench/` branch
- Optional agent: extend the paper's Longevity Claw pattern (GitHub `Insilico-org/longeclaw`) over **your** `masld_kg_v5.gexf` for MASLD-Claw